# Part 1: Supervised Learning - Pneumonia Classification

This section of the CVRIE project focuses on preparing a medical dataset of chest X-rays. The goal is to classify patient records into three distinct categories: Normal (Healthy), Bacteria (Bacterial Pneumonia), or Virus (Viral Pneumonia). 

The following pipeline includes data formatting.

## 1. Library Imports

Before manipulating the dataset, we must import the necessary Python libraries. Here is the utility of each module used in this notebook:

* **`os`**: Allows us to interact with the operating system, navigate through folders, and read file paths.
* **`numpy (np)`**: The core library for scientific computing. It is essential for handling large multi-dimensional arrays (our matrices of pixels) and performing fast mathematical operations.
* **`PIL (Python Imaging Library)`**: Used to open the original images, resize them, and manipulate their color channels.

In [ ]:
import os
import sys
import numpy as np
from PIL import Image

## 2. Dataset Formatting and Classification


Medical images often come in various shapes, sizes, and color formats. To feed them to a Machine Learning model, they must be strictly uniform. In this section, we read the raw images and save a clean, standardized version to a new directory.

To format the dataset we will:
1. **Resize** all images to `256x256` and change color palette to grayscale
2. **Classify** images into an organized directory structure:
   - `dataset/train/normal` — normal images (train)
   - `dataset/train/bacteria` — bacterial pneumonia (train)
   - `dataset/train/virus` — viral pneumonia (train)
   - `dataset/test/normal` — normal images (test)
   - `dataset/test/bacteria` — bacterial pneumonia (test)
   - `dataset/test/virus` — viral pneumonia (test)
3. **Count** images before and after processing to ensure nothing is lost

## `resize_image(image_path, output_path, dim1, dim2)`

Resizes an image to dimensions `dim1 x dim2` and saves it to `output_path`.
It also convert images to 1 canal (only Grey) with the line `img = img.convert('L')`.
after that the dataset is fully uniformised.

**Parameters:**
- `image_path`: path to the source image
- `output_path`: destination path
- `dim1`, `dim2`: target dimensions

**Key lines of code explained:**
* `Image.open()`: Loads the raw image file from the disk.
* `img.convert('L')`: Converts the image strictly to grayscale. Since some X-rays are saved in RGB and others in grayscale, forcing them to 'L' prevents severe dimension mismatch errors during the array creation.
* `img.resize((dim1, dim2))`: Reduces the image size to dim1 x dim2 pixels. This reduces the number of features per image, preventing memory overload and speeding up the model training.

**Returns:** the path of the saved image

In [ ]:
def resize_image(image_path, output_path, dim1, dim2):
    print("Resizing image:", image_path)
    print("Output path:", output_path)
    
    # Creates the destination directory if it does not exist yet to avoid FileNotFoundError
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Loads the raw image file from the hard drive into memory
    img = Image.open(image_path)
    
    # Converts the image strictly to an 8-bit grayscale format ('L').
    # Medical datasets often mix 3-channel RGB images and 1-channel grayscale images.
    # Forcing a single channel prevents severe dimension mismatch errors during matrix creation later.
    img = img.convert('L')
    
    # Resizes the image to the target dimensions (dim1 x dim2, here 256x256).
    # Machine Learning models require inputs of the exact same size. 
    # Downscaling also reduces the number of features, saving RAM and computation time.
    img = img.resize((dim1, dim2))
    
    # Saves the newly formatted, uniformized image to the clean dataset folder
    img.save(output_path)
    return output_path

## `classify_image(filename, source_subdir)`

Determines the destination subdirectory for an image based on its filename and source directory.

**Classification logic:**
- If the image comes from `NORMAL` → `normal` folder
- If the image comes from `PNEUMONIA` and contains **"bacteria"** in its name → `bacteria` folder
- If the image comes from `PNEUMONIA` and contains **"virus"** in its name → `virus` folder
- Otherwise → `None` (image skipped)

**Returns:** the destination subdirectory name (`"normal"`, `"bacteria"`, `"virus"`) or `None`

In [ ]:
def classify_image(filename, source_subdir):
    """Determines the destination folder based on the filename and its source directory."""
    
    # name_lower: Converts the filename to lowercase to avoid case-sensitive bugs (e.g., treating "Virus.jpg" differently from "virus.jpg").
    name_lower = filename.lower()
    
    # If the image was in the 'NORMAL' folder, it clearly belongs to a healthy patient.
    if source_subdir == "NORMAL":
        return "normal"
    # However, if it was in the 'PNEUMONIA' folder, we must read the filename to know WHICH type of pneumonia it is.
    elif source_subdir == "PNEUMONIA":
        # Searches for the specific word inside the filename.
        if "bacteria" in name_lower:
            return "bacteria"
        elif "virus" in name_lower:
            return "virus"
            
    # If the file doesn't match our specific medical rules, we return None so the script ignores it.
    return None

## `count_images(filepath)`

Recursively counts the number of images (`.png`, `.jpg`, `.jpeg`) in a given directory.

**Parameter:** `filepath` — path of the directory to scan

**Returns:** the total number of images found

In [ ]:
def count_images(filepath):
    """counts the number of image files in the dataset folder and its subfolders."""
    count = 0
    # the loop whose parse the dataset folders.
    for root, dirs, filenames in os.walk(filepath):
        for filename in filenames:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                count += 1
    return count

## `format_images(filepath, output_base)`

Main processing function. Iterates through the `train/` and `test/` directories of the source dataset and for each image:
1. Determines its label via `classify_image()`
2. Resizes it to **256x256** via `resize_image()`
3. Saves it to the appropriate destination folder

**Parameters:**
- `filepath`: path to the source dataset (containing `train/` and `test/`)
- `output_base`: root output path (e.g., `./dataset`)

**Returns:** a `np.array` containing the paths of all processed images

In [ ]:
def format_images(filepath, output_base):
    images = []

    # Basic security check: if the provided source folder doesn't exist, we stop the program immediately.
    if not os.path.exists(filepath):
        sys.stderr.write(f"Error: Directory '{filepath}' does not exist.\n")
        sys.exit(84)

    # We iterate through both 'train' (learning data) and 'test' (exam data) folders.
    for split in ["train", "test"]:
        # os.path.join: Safely glues paths together (e.g., "dataset" + "train" = "dataset/train"). 
        # It prevents bugs when switching between Windows (uses \) and Mac/Linux (uses /).
        split_path = os.path.join(filepath, split)
        if not os.path.exists(split_path):
            continue
        
        for category in ["NORMAL", "PNEUMONIA"]:
            category_path = os.path.join(split_path, category)
            if not os.path.exists(category_path):
                continue
            
            for filename in os.listdir(category_path):
                # We strictly filter for image files. If it's a hidden system file, we skip it.
                if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                
                # We call our custom sorting rule to know exactly in which of the 3 new folders this image should go.
                dest_subdir = classify_image(filename, category)
                if dest_subdir is None:
                    continue

                source_path = os.path.join(category_path, filename)
                output_path = os.path.join(output_base, split, dest_subdir, filename)

                # try/except block: Medical datasets can contain corrupted files.
                # Instead of crashing the entire script after 3 hours because of one bad image, 
                # this block catches the error, displays a warning, and moves on to the next patient.
                try:
                    result = resize_image(source_path, output_path, 256, 256)
                    images.append(result)
                except Exception as e:
                    sys.stderr.write(f"Error processing image {filename}: {e}\n")

    return np.array(images) if images else np.array([])

## `main()`

Entry point of the script, runs in this order:
1. **Pre-processing count** — displays the number of images per source directory (`train/NORMAL`, `train/PNEUMONIA`, `test/NORMAL`, `test/PNEUMONIA` and total)
2. **Processing** — calls `format_images` to resize and classify all images
3. **Post-processing count** — displays the number of images per destination directory (`train/normal`, `train/bacteria`, etc.) to verify all images were processed

In [ ]:
def main():
    dataset_folder = os.path.abspath('./chest_xray')
    output_folder = os.path.abspath('./dataset')

    # Count images in the dataset before processing and print the result
    print("IMAGE COUNT BEFORE PROCESSING")
    total_before = 0
    for split in ["train", "test"]:
        split_path = os.path.join(dataset_folder, split)
        if os.path.exists(split_path):
            for category in ["NORMAL", "PNEUMONIA"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_before += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL before: {total_before} images")

    # Process and format the images, then print the number of formatted images
    formatted_images = format_images(dataset_folder, output_folder)
    print(f"\nNumber of formatted images: {len(formatted_images)}")

    # Count images in the output dataset after processing and print the result to see if it matches the number of formatted images
    print("IMAGE COUNT AFTER PROCESSING")
    total_after = 0
    for split in ["train", "test"]:
        split_path = os.path.join(output_folder, split)
        if os.path.exists(split_path):
            for category in ["normal", "bacteria", "virus"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_after += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL after: {total_after} images")

if __name__ == "__main__":
    main()